[Back to Data Structures and Algorithms guideline](Data-Structure&Algorithm.html)


## **Graphs and Core Graph Algorithms** {#graphs-and-core-graph-algorithms}

A graph models **entities and relationships** without requiring a single hierarchy. Roads connect cities, hyperlinks connect pages, prerequisites connect courses, and interactions connect users. Unlike a tree, a graph may contain several paths to the same vertex, cycles, disconnected regions, and direction-dependent reachability.

This chapter follows a deliberate progression:

- the Graph ADT defines which relationships can be queried or changed;
- a representation determines the cost of those operations;
- BFS and DFS provide general ways to explore reachable structure;
- traversal state reveals components, cycles, bipartiteness, and dependency order;
- DSU maintains evolving connectivity without storing paths;
- shortest-path and minimum-spanning-tree algorithms optimize different global objectives.

The most important habit is to state the graph model before selecting an algorithm: directed or undirected, weighted or unweighted, sparse or dense, static or changing. A correct algorithm under one model can be invalid or unnecessarily expensive under another.


### **Graph Terminology and Representations** {#graph-terminology-and-representations}

A **graph** is written as $G=(V,E)$, where $V$ is a set of **vertices** and $E$ is a set of **edges** connecting pairs of vertices. A social network might use people as vertices and friendships as edges; a road network might use intersections as vertices and roads as weighted edges.

An **undirected edge** $\{u,v\}$ can be traversed in either direction. A **directed edge** $(u,v)$ goes from <code>u</code> to <code>v</code> and does not imply the reverse edge. A **weighted graph** attaches a cost, distance, time, or capacity $w(u,v)$ to each edge. The weight is application meaning, not merely decoration: shortest-path algorithms depend on whether weights are nonnegative, while BFS treats every edge as one equal step.

Further vocabulary describes the structure an algorithm can use:

- A **walk** may repeat vertices and edges; a **path** normally contains no repeated vertex.
- A **cycle** starts and ends at the same vertex without otherwise repeating vertices.
- Two vertices are **adjacent** when an edge connects them. The **degree** of an undirected vertex is its number of incident edges.
- In a directed graph, **indegree** counts incoming edges and **outdegree** counts outgoing edges.
- A **simple graph** has no self-loop and no parallel edge. A multigraph may allow both, so its ADT and representation must define how duplicates are stored.
- A graph is **sparse** when $|E|$ is much smaller than $|V|^2$ and **dense** when many possible vertex pairs are connected.

The Graph ADT separates these meanings from storage:

| ADT operation | Required behavior |
|---|---|
| <code>add_vertex(v)</code> | Add <code>v</code> if it is not present. |
| <code>add_edge(u, v, weight)</code> | Add a directed or undirected relationship according to the graph type. |
| <code>remove_vertex(v)</code> | Remove <code>v</code> and all incident edges. |
| <code>remove_edge(u, v)</code> | Remove the specified relationship. |
| <code>has_edge(u, v)</code> | Report whether the relationship exists. |
| <code>neighbors(v)</code> | Iterate over vertices reached by outgoing edges from <code>v</code>. |
| <code>vertices()</code> / <code>edges()</code> | Iterate over the graph elements without duplicating undirected edges. |

![The same graph stored as adjacency lists and as an adjacency matrix.](assets/graph-representations.svg){fig-align="center" width="96%"}

The three most common representations trade memory for different queries:

| Operation or property | Adjacency list | Adjacency matrix | Edge list |
|---|---:|---:|---:|
| storage | $\Theta(|V|+|E|)$ | $\Theta(|V|^2)$ | $\Theta(|V|+|E|)$ |
| test edge $(u,v)$ | $O(\deg(u))$ in a list; expected $O(1)$ with a hash set | $O(1)$ | $O(|E|)$ |
| iterate neighbors of <code>u</code> | $\Theta(\deg(u))$ | $\Theta(|V|)$ | $\Theta(|E|)$ |
| insert edge | $O(1)$ amortized | $O(1)$ | $O(1)$ amortized |
| remove edge | $O(\deg(u))$ in a list; expected $O(1)$ with a hash set | $O(1)$ | $O(|E|)$ |
| iterate all edges | $\Theta(|V|+|E|)$ | $\Theta(|V|^2)$ | $\Theta(|E|)$ |
| best fit | sparse graphs and traversal | dense graphs and constant-time edge tests | edge-centric algorithms such as Kruskal |

Here $|V|$ and $|E|$ are vertex and edge counts, and $\deg(u)$ is the number of neighbors stored for <code>u</code>. An undirected adjacency list stores each edge twice, once at each endpoint, but this is still $\Theta(|V|+|E|)$ space. A matrix uses one cell for every possible ordered pair whether or not an edge exists.

![A concrete graph and its adjacency-list memory layout.](assets/graph-adjacency-list.svg){fig-align="center" width="72%"}

*Open visual source: [Wikimedia Commons - Liste di adiacenza](https://commons.wikimedia.org/wiki/File:Liste_di_adiacenza.svg) (CC BY-SA 3.0).*

<details>
<summary>Python implementation: hash-based adjacency-list Graph ADT</summary>

~~~python
from collections.abc import Hashable, Iterator
from typing import Generic, TypeVar

V = TypeVar("V", bound=Hashable)


class Graph(Generic[V]):
    """A simple unweighted graph backed by sets of outgoing neighbors."""

    def __init__(self, directed: bool = False) -> None:
        self.directed = directed
        self._adjacency: dict[V, set[V]] = {}

    def add_vertex(self, vertex: V) -> None:
        self._adjacency.setdefault(vertex, set())

    def add_edge(self, source: V, target: V) -> None:
        self.add_vertex(source)
        self.add_vertex(target)
        self._adjacency[source].add(target)

        if not self.directed:
            # An undirected relationship is represented in both lists.
            self._adjacency[target].add(source)

    def remove_edge(self, source: V, target: V) -> None:
        self._adjacency[source].remove(target)
        if not self.directed:
            self._adjacency[target].remove(source)

    def has_edge(self, source: V, target: V) -> bool:
        return target in self._adjacency.get(source, set())

    def neighbors(self, vertex: V) -> frozenset[V]:
        """Return a read-only snapshot so callers cannot break symmetry."""
        return frozenset(self._adjacency[vertex])

    def vertices(self) -> Iterator[V]:
        return iter(self._adjacency)


network: Graph[str] = Graph(directed=False)
network.add_edge("A", "B")
network.add_edge("A", "C")
network.add_edge("C", "D")

assert network.has_edge("B", "A")
assert network.neighbors("A") == frozenset({"B", "C"})
network.remove_edge("A", "B")
assert not network.has_edge("B", "A")
~~~

</details>

Using neighbor sets gives expected $O(1)$ edge tests and updates, at the cost of hash-table overhead and no intrinsic neighbor order. Lists use less overhead and preserve insertion order but may scan a whole neighbor list to test or remove an edge.

**Practice.** [LeetCode 133 - Clone Graph](https://leetcode.com/problems/clone-graph/) exercises graph representation, identity tracking, and reconstruction of every adjacency relationship.


### **Breadth-First Search and Depth-First Search** {#breadth-first-search-and-depth-first-search}

Graph traversal asks: starting from vertex <code>s</code>, which vertices are reachable, and in what order should their outgoing edges be explored? Unlike a tree, a graph can return to an already encountered vertex. Every correct traversal therefore maintains a **visited set** so a cycle does not cause infinite repetition and a vertex reached by several paths is processed once.

**Breadth-first search (BFS)** explores vertices in nondecreasing number of edges from the start. Its FIFO queue stores the discovered but not yet expanded frontier. Marking a vertex when it is enqueued, rather than when it is later removed, prevents several parents from inserting duplicate copies.

**Depth-first search (DFS)** follows one outgoing path until it cannot continue, then backtracks. A recursive call stack or explicit LIFO stack stores the unfinished path. DFS is useful when finish order and nested exploration matter; BFS is useful when distance in unweighted edges or level order matters.

~~~text
BFS(graph, start)
    visited <- {start}
    queue <- FIFO queue containing start
    while queue is not empty
        u <- remove_front(queue)
        visit(u)
        for each v in graph.neighbors(u)
            if v is not in visited
                add v to visited
                add_back(queue, v)

DFS(graph, u, visited)
    add u to visited
    visit(u)
    for each v in graph.neighbors(u)
        if v is not in visited
            DFS(graph, v, visited)
~~~

::: {layout-ncol=2}
![BFS expands the frontier one distance layer at a time.](assets/graph-bfs.gif){width="82%" fig-align="center"}

![DFS follows one branch and then backtracks.](assets/graph-dfs.gif){width="82%" fig-align="center"}
:::

*Open visual sources: [Wikimedia Commons - Breadth-First-Search-Algorithm.gif](https://commons.wikimedia.org/wiki/File:Breadth-First-Search-Algorithm.gif) and [Depth-First-Search.gif](https://commons.wikimedia.org/wiki/File:Depth-First-Search.gif) (CC BY-SA / GFDL).*

With adjacency lists, each reached vertex is processed once and every outgoing adjacency entry is examined once. A complete traversal therefore takes $O(|V|+|E|)$ time and $O(|V|)$ auxiliary space for visited state plus the frontier. In an undirected graph each edge appears in two lists, but $2|E|$ remains $O(|E|)$. With an adjacency matrix, finding every vertex's neighbors scans a full row, so traversal takes $O(|V|^2)$ time.

The maximum BFS queue can contain $O(|V|)$ vertices in a wide frontier. Recursive DFS uses $O(|V|)$ stack space in the worst case of a long path and can exceed a language recursion limit; an explicit stack is safer for large graphs.

<details>
<summary>Python implementation: deterministic BFS and iterative DFS</summary>

~~~python
from collections import deque
from collections.abc import Mapping, Sequence


def bfs(graph: Mapping[str, Sequence[str]], start: str) -> list[str]:
    """Visit reachable vertices in nondecreasing edge distance."""
    order: list[str] = []
    visited = {start}
    queue = deque([start])

    while queue:
        vertex = queue.popleft()
        order.append(vertex)

        for neighbor in graph.get(vertex, ()):
            if neighbor not in visited:
                visited.add(neighbor)       # Mark at discovery time.
                queue.append(neighbor)

    return order


def dfs_iterative(
    graph: Mapping[str, Sequence[str]], start: str
) -> list[str]:
    """Visit reachable vertices depth-first without recursion."""
    order: list[str] = []
    visited: set[str] = set()
    stack = [start]

    while stack:
        vertex = stack.pop()
        if vertex in visited:
            continue

        visited.add(vertex)
        order.append(vertex)

        # Reverse so the first listed neighbor is processed first by LIFO.
        for neighbor in reversed(graph.get(vertex, ())):
            if neighbor not in visited:
                stack.append(neighbor)

    return order


graph = {
    "A": ["B", "C"],
    "B": ["D", "E"],
    "C": ["F"],
    "D": [], "E": ["F"], "F": [],
}

assert bfs(graph, "A") == ["A", "B", "C", "D", "E", "F"]
assert dfs_iterative(graph, "A") == ["A", "B", "D", "E", "F", "C"]
~~~

</details>

BFS and DFS answer reachability, but their traversal trees are not unique when neighbor order is unspecified. Correctness should depend on which vertices or structural properties are found, not on one accidental ordering.

**Practice.** [LeetCode 1971 - Find if Path Exists in Graph](https://leetcode.com/problems/find-if-path-exists-in-graph/) is a direct exercise in building an adjacency list and stopping a traversal when the destination is reached.


### **Connectivity and Connected Components** {#connectivity-and-connected-components}

In an undirected graph, vertices <code>u</code> and <code>v</code> are **connected** when some path joins them. Connectivity partitions the vertices into maximal **connected components**: vertices inside one component can reach each other, and no edge joins two different components. This is useful for identifying isolated networks, counting clusters, checking whether all locations communicate, or solving grid-island problems after modeling cells as vertices.

A single BFS or DFS finds the component containing its start vertex. To find every component, scan all vertices and launch a traversal whenever an unvisited vertex is encountered:

~~~text
CONNECTED-COMPONENTS(graph)
    visited <- empty set
    components <- empty list
    for each vertex v
        if v is not in visited
            component <- traverse from v
            add every reached vertex to visited
            append component to components
    return components
~~~

Direction makes connectivity less symmetric. A directed graph is **weakly connected** if replacing every directed edge with an undirected edge connects it. A **strongly connected component (SCC)** is a maximal set in which every vertex can reach every other vertex by directed paths. Collapsing each SCC to one super-vertex always produces a directed acyclic graph, called the condensation graph.

![Undirected components partition reachability directly; directed SCCs condense into a DAG.](assets/connected-components-scc.svg){fig-align="center" width="96%"}

Kosaraju's SCC algorithm uses DFS finish order:

~~~text
KOSARAJU(graph)
    run DFS on graph and record each vertex when it finishes
    transpose graph by reversing every edge
    process vertices on the transpose in decreasing finish order
    each new DFS tree is one strongly connected component
~~~

The first pass orders SCCs so the second pass cannot leak from the next selected component into an unprocessed predecessor component. Both passes inspect every vertex and edge, so connected components and Kosaraju SCC decomposition take $O(|V|+|E|)$ time with adjacency lists and $O(|V|)$ auxiliary traversal state, in addition to $O(|V|+|E|)$ for an explicitly stored transpose.

<details>
<summary>Python implementation: undirected components and directed SCCs</summary>

~~~python
from collections.abc import Mapping, Sequence


def connected_components(
    graph: Mapping[str, Sequence[str]]
) -> list[set[str]]:
    """Partition an undirected graph into reachable components."""
    visited: set[str] = set()
    result: list[set[str]] = []

    for start in graph:
        if start in visited:
            continue

        component: set[str] = set()
        stack = [start]
        visited.add(start)

        while stack:
            vertex = stack.pop()
            component.add(vertex)
            for neighbor in graph[vertex]:
                if neighbor not in visited:
                    visited.add(neighbor)
                    stack.append(neighbor)

        result.append(component)

    return result


def strongly_connected_components(
    graph: Mapping[str, Sequence[str]]
) -> list[set[str]]:
    """Return SCCs of a directed graph using Kosaraju's two passes."""
    vertices = set(graph)
    for neighbors in graph.values():
        vertices.update(neighbors)

    visited: set[str] = set()
    finish_order: list[str] = []

    def record_finish(vertex: str) -> None:
        visited.add(vertex)
        for neighbor in graph.get(vertex, ()):
            if neighbor not in visited:
                record_finish(neighbor)
        finish_order.append(vertex)          # Append after descendants finish.

    for vertex in vertices:
        if vertex not in visited:
            record_finish(vertex)

    transpose: dict[str, list[str]] = {vertex: [] for vertex in vertices}
    for source in vertices:
        for target in graph.get(source, ()):
            transpose[target].append(source)

    visited.clear()
    components: list[set[str]] = []

    for start in reversed(finish_order):
        if start in visited:
            continue
        component: set[str] = set()
        stack = [start]
        visited.add(start)

        while stack:
            vertex = stack.pop()
            component.add(vertex)
            for neighbor in transpose[vertex]:
                if neighbor not in visited:
                    visited.add(neighbor)
                    stack.append(neighbor)
        components.append(component)

    return components


undirected = {
    "A": ["B"], "B": ["A"],
    "C": ["D"], "D": ["C"],
    "E": [],
}
assert {frozenset(c) for c in connected_components(undirected)} == {
    frozenset({"A", "B"}), frozenset({"C", "D"}), frozenset({"E"})
}

directed = {
    "A": ["B"], "B": ["A", "C"],
    "C": ["D"], "D": ["C", "E"], "E": [],
}
assert {frozenset(c) for c in strongly_connected_components(directed)} == {
    frozenset({"A", "B"}), frozenset({"C", "D"}), frozenset({"E"})
}
~~~

</details>

For static graphs, traversal returns the actual members and can recover paths. For a stream of edge additions dominated by repeated “are these connected?” queries, the Disjoint Set Union structure introduced later is usually more efficient.

**Practice.** [LeetCode 547 - Number of Provinces](https://leetcode.com/problems/number-of-provinces/) trains component counting from an adjacency matrix rather than an adjacency list.


### **Cycle Detection and Bipartite Graphs** {#cycle-detection-and-bipartite-graphs}

A cycle means a traversal can return to an earlier vertex, but the evidence for a cycle depends on edge direction. In an undirected DFS, immediately seeing the parent edge is expected because every edge is stored in both directions; a cycle exists only when an explored vertex has an already visited neighbor different from its parent. In a directed DFS, an edge to a vertex that is still active on the current recursion path is a **back edge** and proves a directed cycle.

Directed cycle detection therefore uses three states:

- **white:** unseen;
- **gray:** entered but not finished, so it lies on the active DFS path;
- **black:** completely processed.

~~~text
HAS-DIRECTED-CYCLE(graph)
    initially color every vertex white

    DFS(u)
        color u gray
        for each edge u -> v
            if v is gray
                return true
            if v is white and DFS(v) is true
                return true
        color u black
        return false

    run DFS from every remaining white vertex
~~~

A graph is **bipartite** when its vertices can be divided into sets $L$ and $R$ so every edge has one endpoint in each set. Equivalently, an undirected graph is bipartite exactly when it contains no odd-length cycle. BFS or DFS can attempt a two-coloring: assign the start one color, give every neighbor the opposite color, and reject an edge whose endpoints receive the same color.

~~~text
IS-BIPARTITE(graph)
    color <- empty map
    for each uncolored start vertex
        color[start] <- blue
        run BFS from start
            for each edge u -- v
                if v is uncolored
                    color[v] <- opposite(color[u])
                else if color[v] equals color[u]
                    return false
    return true
~~~

![Directed DFS detects an edge to an active gray ancestor; bipartite testing rejects a same-color edge on an odd cycle.](assets/cycle-bipartite.svg){fig-align="center" width="96%"}

Both tests take $O(|V|+|E|)$ time with adjacency lists and $O(|V|)$ state. The crucial distinction is the invariant: directed cycle detection tracks the current DFS path, while bipartite testing tracks a parity assignment that must remain consistent along every edge.

Cycle detection appears in dependency validation, deadlock analysis, and topological sorting. Bipartite structure appears in two-group assignment, matching problems, and checking whether pairwise conflicts can be separated into two sides.

<details>
<summary>Python implementation: directed cycles and bipartite coloring</summary>

~~~python
from collections import deque
from collections.abc import Mapping, Sequence


def has_directed_cycle(graph: Mapping[str, Sequence[str]]) -> bool:
    """Detect a back edge with white/gray/black DFS state."""
    WHITE, GRAY, BLACK = 0, 1, 2
    vertices = set(graph)
    for neighbors in graph.values():
        vertices.update(neighbors)
    color = {vertex: WHITE for vertex in vertices}

    def visit(vertex: str) -> bool:
        color[vertex] = GRAY
        for neighbor in graph.get(vertex, ()):
            if color[neighbor] == GRAY:
                return True                  # Edge returns to active path.
            if color[neighbor] == WHITE and visit(neighbor):
                return True
        color[vertex] = BLACK
        return False

    return any(color[v] == WHITE and visit(v) for v in vertices)


def is_bipartite(graph: Mapping[str, Sequence[str]]) -> bool:
    """Try to assign opposite Boolean colors across every edge."""
    vertices = set(graph)
    for neighbors in graph.values():
        vertices.update(neighbors)
    color: dict[str, bool] = {}

    for start in vertices:                    # Handles disconnected graphs.
        if start in color:
            continue
        color[start] = False
        queue = deque([start])

        while queue:
            vertex = queue.popleft()
            for neighbor in graph.get(vertex, ()):
                if neighbor not in color:
                    color[neighbor] = not color[vertex]
                    queue.append(neighbor)
                elif color[neighbor] == color[vertex]:
                    return False
    return True


cyclic = {"A": ["B"], "B": ["C"], "C": ["A"]}
acyclic = {"A": ["B", "C"], "B": ["D"], "C": ["D"], "D": []}
assert has_directed_cycle(cyclic)
assert not has_directed_cycle(acyclic)

square = {"A": ["B", "D"], "B": ["A", "C"],
          "C": ["B", "D"], "D": ["A", "C"]}
triangle = {"A": ["B", "C"], "B": ["A", "C"], "C": ["A", "B"]}
assert is_bipartite(square)
assert not is_bipartite(triangle)
~~~

</details>

**Practice.** [LeetCode 785 - Is Graph Bipartite?](https://leetcode.com/problems/is-graph-bipartite/) requires two-coloring every disconnected component and detecting the first parity conflict.


### **Topological Sorting** {#topological-sorting}

A **topological ordering** of a directed graph is a linear sequence in which every edge $u\rightarrow v$ places <code>u</code> before <code>v</code>. It models dependencies: a prerequisite before its course, a build target after its inputs, or a task after every required predecessor. Such an ordering exists exactly when the graph is a **directed acyclic graph (DAG)**.

The ordering is not necessarily unique. If two currently available tasks have no dependency relation, either may appear first. A topological algorithm promises to respect all edges, not to reproduce one fixed sequence unless a tie-breaking rule is also specified.

Kahn's algorithm maintains each vertex's remaining indegree. A vertex with indegree zero has no unmet prerequisite and can safely be emitted. Removing its outgoing edges may make further vertices available:

~~~text
KAHN-TOPOLOGICAL-SORT(graph)
    indegree[v] <- number of incoming edges for every vertex v
    ready <- queue containing every vertex whose indegree is zero
    order <- empty list

    while ready is not empty
        u <- remove_front(ready)
        append u to order
        for each edge u -> v
            indegree[v] <- indegree[v] - 1
            if indegree[v] equals zero
                add_back(ready, v)

    if length(order) is less than |V|
        report a directed cycle
    return order
~~~

![In a topological ordering of a DAG, every directed edge points from an earlier vertex to a later vertex.](assets/topological-ordering.svg){fig-align="center" width="48%"}

*Open visual source: [Wikimedia Commons - Topological Ordering.svg](https://commons.wikimedia.org/wiki/File:Topological_Ordering.svg) (CC0).*

Why does the final length detect a cycle? Every nonempty DAG contains at least one zero-indegree vertex. If vertices remain but none has zero indegree, following predecessor edges inside the remaining finite subgraph must eventually repeat a vertex, proving a directed cycle.

Kahn's algorithm takes $O(|V|+|E|)$ time and $O(|V|)$ auxiliary space with adjacency lists. DFS provides an alternative: append a vertex after all descendants finish and reverse the finish order, while gray-state back-edge detection rejects cycles. Kahn's version is often clearer when prerequisite counts or parallel scheduling batches matter.

<details>
<summary>Python implementation: Kahn's algorithm with cycle reporting</summary>

~~~python
from collections import deque
from collections.abc import Mapping, Sequence


def topological_sort(
    graph: Mapping[str, Sequence[str]]
) -> list[str] | None:
    """Return a valid order, or None when the directed graph has a cycle."""
    # Preserve first-seen order while including target-only vertices.
    vertices = list(graph)
    known = set(vertices)
    for neighbors in graph.values():
        for neighbor in neighbors:
            if neighbor not in known:
                known.add(neighbor)
                vertices.append(neighbor)

    indegree = {vertex: 0 for vertex in vertices}
    for source in vertices:
        for target in graph.get(source, ()):
            indegree[target] += 1

    ready = deque(v for v in vertices if indegree[v] == 0)
    order: list[str] = []

    while ready:
        source = ready.popleft()
        order.append(source)

        for target in graph.get(source, ()):
            indegree[target] -= 1            # One prerequisite is complete.
            if indegree[target] == 0:
                ready.append(target)

    return order if len(order) == len(vertices) else None


courses = {
    "Foundations": ["Algorithms", "Databases"],
    "Algorithms": ["Advanced Algorithms"],
    "Databases": ["Project"],
    "Advanced Algorithms": ["Project"],
    "Project": [],
}
order = topological_sort(courses)
assert order is not None
position = {course: i for i, course in enumerate(order)}
assert all(
    position[source] < position[target]
    for source, targets in courses.items()
    for target in targets
)

assert topological_sort({"A": ["B"], "B": ["A"]}) is None
~~~

</details>

**Practice.** [LeetCode 210 - Course Schedule II](https://leetcode.com/problems/course-schedule-ii/) is a direct dependency-ordering problem and requires returning an empty result when no topological order exists.


### **Disjoint Set Union** {#disjoint-set-union}

**Disjoint Set Union (DSU)**, also called **Union-Find**, maintains a partition of elements into non-overlapping sets. It answers a narrow but important question efficiently: “are these two elements currently in the same component?” Imagine initially placing every vertex on its own island, then joining islands whenever an edge arrives.

DSU does not store graph paths or neighbor lists. Each set is represented by a rooted parent forest, and one root acts as the set's representative. That narrower representation is why DSU is better than rerunning BFS after every edge insertion when the workload consists of repeated unions and connectivity queries.

The DSU ADT provides:

| ADT operation | Required behavior |
|---|---|
| <code>make_set(x)</code> | Create the singleton set $\{x\}$. |
| <code>find(x)</code> | Return the representative of the set containing <code>x</code>. |
| <code>union(x, y)</code> | Merge the two sets if their representatives differ. |
| <code>connected(x, y)</code> | Report whether <code>find(x) == find(y)</code>. |
| <code>component_size(x)</code> | Return the size stored at <code>x</code>'s representative. |
| <code>component_count()</code> | Return the number of current disjoint sets. |

Two structural optimizations make the forest extremely shallow:

- **union by size or rank** attaches the smaller tree's root below the larger tree's root;
- **path compression** rewrites parent links encountered by <code>find</code> so those vertices point close to, or directly at, the representative.

~~~text
FIND(x)
    if parent[x] is not x
        parent[x] <- FIND(parent[x])
    return parent[x]

UNION(x, y)
    root_x <- FIND(x)
    root_y <- FIND(y)
    if root_x equals root_y
        return false
    attach the smaller root below the larger root
    update the larger root's size
    return true
~~~

![Path compression preserves the representative while replacing a long find path with direct root links.](assets/dsu-path-compression.svg){fig-align="center" width="96%"}

With both optimizations, any sequence of <code>m</code> operations on <code>n</code> elements takes $O(m\alpha(n))$ time. The function $\alpha(n)$ is the inverse Ackermann function and grows so slowly that it is at most a tiny constant for any practical input. Thus <code>find</code>, <code>union</code>, and <code>connected</code> have amortized $O(\alpha(n))$ cost, while initialization and storage are $\Theta(n)$.

DSU supports edge additions well but not arbitrary edge deletions: removing one edge may split a component, and a representative forest contains too little information to discover the two resulting sides. Traversal remains preferable when paths, component members, or a changing graph with deletions must be recovered.

<details>
<summary>Python implementation: union by size with path halving</summary>

~~~python
class DisjointSet:
    def __init__(self, size: int) -> None:
        if size < 0:
            raise ValueError("size must be non-negative")
        self.parent = list(range(size))
        self.size = [1] * size
        self.components = size

    def find(self, item: int) -> int:
        """Return the representative while shortening the search path."""
        while item != self.parent[item]:
            # Path halving points each visited node to its grandparent.
            self.parent[item] = self.parent[self.parent[item]]
            item = self.parent[item]
        return item

    def union(self, first: int, second: int) -> bool:
        """Merge two components; return False if already connected."""
        root_a = self.find(first)
        root_b = self.find(second)
        if root_a == root_b:
            return False

        # Make root_a the larger root before attaching root_b.
        if self.size[root_a] < self.size[root_b]:
            root_a, root_b = root_b, root_a

        self.parent[root_b] = root_a
        self.size[root_a] += self.size[root_b]
        self.components -= 1
        return True

    def connected(self, first: int, second: int) -> bool:
        return self.find(first) == self.find(second)

    def component_size(self, item: int) -> int:
        return self.size[self.find(item)]


dsu = DisjointSet(6)
assert dsu.union(0, 1)
assert dsu.union(1, 2)
assert not dsu.union(0, 2)             # This edge would be redundant.
assert dsu.union(3, 4)
assert dsu.connected(0, 2)
assert not dsu.connected(0, 4)
assert dsu.component_size(1) == 3
assert dsu.components == 3             # {0,1,2}, {3,4}, {5}
~~~

</details>

**Practice.** [LeetCode 684 - Redundant Connection](https://leetcode.com/problems/redundant-connection/) uses DSU to identify the first edge whose endpoints already share a representative and would therefore close a cycle.


### **Shortest Paths** {#shortest-paths}

For a weighted graph, the length of a path is the sum of its edge weights. The **single-source shortest-path problem** asks for the minimum path distance from a source <code>s</code> to every reachable vertex. The answer is not the path with the fewest edges unless all edges have equal weight: a route with more roads may have a smaller total cost.

Shortest-path algorithms maintain a tentative distance $d[v]$. Initially $d[s]=0$ and every other distance is infinity. **Relaxing** edge $(u,v)$ with weight $w(u,v)$ tests whether reaching <code>v</code> through <code>u</code> improves the current best route:

$$
d[v] \leftarrow \min\bigl(d[v],\ d[u]+w(u,v)\bigr).
$$

Here $d[u]$ is the best known source-to-<code>u</code> distance and $w(u,v)$ is the edge cost. If the second expression is smaller, the algorithm also records <code>parent[v] = u</code>, allowing the final path to be reconstructed backward from the destination.

The correct algorithm depends on the weight model:

| Graph condition | Algorithm | Time with suitable representation | Key idea |
|---|---|---:|---|
| unweighted or every edge has equal cost | BFS | $O(|V|+|E|)$ | queue explores by edge distance |
| DAG with any edge weights | topological relaxation | $O(|V|+|E|)$ | every predecessor is final before its outgoing edges |
| nonnegative edge weights | Dijkstra | $O((|V|+|E|)\log |V|)$ with a binary heap | settle smallest tentative distance |
| negative edges but no reachable negative cycle | Bellman-Ford | $O(|V||E|)$ | relax every edge up to $|V|-1$ rounds |
| all-pairs distances | Floyd-Warshall | $O(|V|^3)$ time, $O(|V|^2)$ space | dynamic programming over allowed intermediate vertices |

Dijkstra's algorithm uses a min-priority queue whose entries are <code>(distance, vertex)</code>. It repeatedly settles the unsettled vertex with the smallest tentative distance:

~~~text
DIJKSTRA(graph, source)
    distance[source] <- 0; every other distance <- infinity
    frontier <- min-priority queue containing (0, source)

    while frontier is not empty
        dist_u, u <- extract_min(frontier)
        if dist_u is stale
            continue
        for each weighted edge u -> v
            candidate <- dist_u + weight(u, v)
            if candidate < distance[v]
                distance[v] <- candidate
                parent[v] <- u
                insert (candidate, v) into frontier
~~~

![Dijkstra repeatedly expands the unsettled vertex with minimum tentative distance and relaxes its outgoing edges.](assets/dijkstra-animation.gif){fig-align="center" width="42%"}

*Open visual source: [Wikimedia Commons - Dijkstra Animation.gif](https://commons.wikimedia.org/wiki/File:Dijkstra_Animation.gif) (public domain).*

Nonnegative weights justify the greedy settlement step. If <code>u</code> has the smallest tentative distance, any alternative path that reaches <code>u</code> later must first pass through a vertex with at least that tentative distance and then add a nonnegative edge; it cannot improve <code>u</code>. A negative edge destroys this argument because a path discovered later may reduce a supposedly final distance.

Python's <code>heapq</code> has no decrease-key operation. The implementation can push a new entry whenever a distance improves and discard old entries when popped. Each successful relaxation may add one heap entry, producing $O((|V|+|E|)\log |V|)$ time, often written $O(|E|\log |V|)$ for a connected sparse graph, and $O(|V|+|E|)$ storage including the graph.

<details>
<summary>Python implementation: Dijkstra with path reconstruction</summary>

~~~python
import heapq
import math
from collections.abc import Mapping, Sequence

WeightedGraph = Mapping[str, Sequence[tuple[str, float]]]


def dijkstra(
    graph: WeightedGraph, start: str
) -> tuple[dict[str, float], dict[str, str | None]]:
    """Return shortest distances and predecessor links from start."""
    vertices = set(graph)
    for edges in graph.values():
        vertices.update(target for target, _ in edges)

    distance = {vertex: math.inf for vertex in vertices}
    parent: dict[str, str | None] = {vertex: None for vertex in vertices}
    distance[start] = 0.0
    frontier: list[tuple[float, str]] = [(0.0, start)]

    while frontier:
        dist_u, source = heapq.heappop(frontier)
        if dist_u != distance[source]:
            continue                         # Ignore an outdated heap entry.

        for target, weight in graph.get(source, ()):
            if weight < 0:
                raise ValueError("Dijkstra requires nonnegative weights")

            candidate = dist_u + weight
            if candidate < distance[target]:
                distance[target] = candidate
                parent[target] = source
                heapq.heappush(frontier, (candidate, target))

    return distance, parent


def reconstruct_path(
    parent: Mapping[str, str | None], start: str, target: str
) -> list[str]:
    path: list[str] = []
    current: str | None = target

    while current is not None:
        path.append(current)
        if current == start:
            return list(reversed(path))
        current = parent[current]
    return []                                  # Target was unreachable.


graph = {
    "A": [("B", 4), ("C", 1)],
    "B": [("D", 1)],
    "C": [("B", 2), ("D", 5)],
    "D": [],
}
distance, parent = dijkstra(graph, "A")
assert distance["D"] == 4
assert reconstruct_path(parent, "A", "D") == ["A", "C", "B", "D"]
~~~

</details>

**Practice.** [LeetCode 743 - Network Delay Time](https://leetcode.com/problems/network-delay-time/) is a direct single-source, nonnegative weighted shortest-path problem and tests how unreachable vertices are handled.


### **Minimum Spanning Trees** {#minimum-spanning-trees}

For a connected, undirected, weighted graph, a **spanning tree** includes every vertex, remains connected, and contains no cycle. Any spanning tree on $|V|$ vertices has exactly $|V|-1$ edges. A **minimum spanning tree (MST)** minimizes the sum of those selected edge weights.

An MST answers “what is the cheapest network that connects everything?” It is appropriate for cable layout, road planning, clustering, and network backbone design. It is not a shortest-path tree: an MST minimizes total selected infrastructure, while a shortest-path tree minimizes routes from one particular source. The path between two vertices inside an MST may be longer than their shortest path in the original graph.

Two exchange principles justify the standard greedy algorithms:

- **Cut property:** for any partition of vertices into two sides, a lightest edge crossing that cut is safe for some MST.
- **Cycle property:** in a cycle, a strictly heaviest edge is unnecessary for at least one MST because the remaining cycle edges still connect its endpoints.

Kruskal's algorithm grows a forest globally. It sorts all edges by nondecreasing weight and uses DSU to accept an edge only when its endpoints lie in different current components:

~~~text
KRUSKAL(vertices, edges)
    create one DSU set per vertex
    selected <- empty list
    for edge (u, v) in nondecreasing weight order
        if FIND(u) differs from FIND(v)
            add edge to selected
            UNION(u, v)
        if selected contains |V| - 1 edges
            stop
    if fewer than |V| - 1 edges were selected
        report that the graph is disconnected
~~~

![Kruskal processes edges from lightest to heaviest, rejects cycle-forming edges, and stops after selecting |V|-1 edges.](assets/mst-kruskal-steps.svg){fig-align="center" width="96%"}

Prim's algorithm grows one connected tree from a chosen start. Its min-heap stores edges crossing the current cut from visited to unvisited vertices:

~~~text
PRIM(graph, start)
    visited <- {start}
    frontier <- every edge leaving start in a min-heap
    while frontier is not empty and not every vertex is visited
        edge (u, v) <- extract minimum crossing edge
        if v is already visited
            continue
        accept (u, v); add v to visited
        insert every edge from v to an unvisited neighbor
~~~

Kruskal with sorting takes $O(|E|\log |E|)$ time, which is equivalent to $O(|E|\log |V|)$ for a simple connected graph, plus almost-linear DSU operations. Heap-based Prim takes $O(|E|\log |V|)$ with adjacency lists. On a dense adjacency matrix, a simple $O(|V|^2)$ Prim implementation may be preferable because it avoids processing a very large heap. Both use $O(|V|+|E|)$ storage with adjacency lists.

Equal weights may produce several different MSTs with the same total weight. If the graph is disconnected, the algorithms produce a **minimum spanning forest** unless the interface explicitly requires failure.

<details>
<summary>Python implementation: Kruskal and Prim under the same graph</summary>

~~~python
import heapq
from collections.abc import Mapping, Sequence

Edge = tuple[float, str, str]          # (weight, source, target)


class DisjointSet:
    def __init__(self, vertices: Sequence[str]) -> None:
        self.parent = {vertex: vertex for vertex in vertices}
        self.size = {vertex: 1 for vertex in vertices}

    def find(self, vertex: str) -> str:
        if self.parent[vertex] != vertex:
            self.parent[vertex] = self.find(self.parent[vertex])
        return self.parent[vertex]

    def union(self, first: str, second: str) -> bool:
        root_a, root_b = self.find(first), self.find(second)
        if root_a == root_b:
            return False
        if self.size[root_a] < self.size[root_b]:
            root_a, root_b = root_b, root_a
        self.parent[root_b] = root_a
        self.size[root_a] += self.size[root_b]
        return True


def kruskal(
    vertices: Sequence[str], edges: Sequence[Edge]
) -> tuple[float, list[Edge]] | None:
    dsu = DisjointSet(vertices)
    selected: list[Edge] = []
    total = 0.0

    for edge in sorted(edges):              # Tuples sort by weight first.
        weight, source, target = edge
        if dsu.union(source, target):        # Rejects cycle-forming edges.
            selected.append(edge)
            total += weight
            if len(selected) == len(vertices) - 1:
                return total, selected

    return (0.0, []) if not vertices else None


def prim(
    graph: Mapping[str, Sequence[tuple[str, float]]], start: str
) -> tuple[float, list[Edge]] | None:
    visited = {start}
    frontier: list[Edge] = [
        (weight, start, target) for target, weight in graph[start]
    ]
    heapq.heapify(frontier)
    selected: list[Edge] = []
    total = 0.0

    while frontier and len(visited) < len(graph):
        weight, source, target = heapq.heappop(frontier)
        if target in visited:
            continue

        visited.add(target)
        selected.append((weight, source, target))
        total += weight

        for neighbor, next_weight in graph[target]:
            if neighbor not in visited:
                heapq.heappush(frontier, (next_weight, target, neighbor))

    return (total, selected) if len(visited) == len(graph) else None


vertices = ["A", "B", "C", "D", "E"]
edges: list[Edge] = [
    (1, "A", "B"), (2, "C", "D"), (3, "B", "C"),
    (4, "A", "C"), (5, "D", "E"), (6, "B", "D"), (7, "C", "E"),
]
adjacency = {vertex: [] for vertex in vertices}
for weight, source, target in edges:
    adjacency[source].append((target, weight))
    adjacency[target].append((source, weight))

kruskal_result = kruskal(vertices, edges)
prim_result = prim(adjacency, "A")
assert kruskal_result is not None and kruskal_result[0] == 11
assert prim_result is not None and prim_result[0] == 11
assert len(kruskal_result[1]) == len(vertices) - 1
~~~

</details>

Kruskal is often convenient when edges already arrive as a list or when the graph is sparse. Prim is often natural with adjacency lists and when the algorithm should grow outward from an existing connected region.

**Practice.** [LeetCode 1584 - Min Cost to Connect All Points](https://leetcode.com/problems/min-cost-to-connect-all-points/) asks for the MST of a complete graph whose edge weights are Manhattan distances.


### **Comparison and Selection** {#comparison-and-selection}

Graph problems become easier to classify when the required output is stated before the algorithm is chosen.

| Required result | Appropriate starting point | Required assumption | Main complexity |
|---|---|---|---:|
| store and traverse a sparse graph | adjacency list | none | $\Theta(|V|+|E|)$ storage |
| constant-time arbitrary edge tests in a dense graph | adjacency matrix | enough memory for all pairs | $\Theta(|V|^2)$ storage |
| reachability or unweighted shortest paths | BFS | equal edge cost for shortest distance | $O(|V|+|E|)$ |
| recursive structure, finish order, or back edges | DFS | visited / color state maintained correctly | $O(|V|+|E|)$ |
| partition an undirected static graph | connected-component traversal | full graph available | $O(|V|+|E|)$ |
| maintain connectivity under edge additions | DSU | deletions and path recovery are not required | amortized $O(\alpha(|V|))$ per operation |
| dependency order | topological sorting | directed acyclic graph | $O(|V|+|E|)$ |
| shortest paths with nonnegative weights | Dijkstra | every reachable edge weight is nonnegative | $O((|V|+|E|)\log |V|)$ |
| shortest paths with negative edges | Bellman-Ford | no reachable negative cycle for finite answers | $O(|V||E|)$ |
| cheapest total connection of all vertices | Kruskal or Prim | connected, undirected weighted graph | $O(|E|\log |V|)$ |

A reliable workflow is:

1. Define vertices and edges, including direction and weight meaning.
2. Decide whether the graph is sparse or dense and select a representation accordingly.
3. State the output: one path, all distances, components, an ordering, or a selected edge set.
4. Identify the invariant that makes the algorithm correct: BFS layer distance, DFS active path, zero remaining indegree, DSU representative, minimum tentative distance, or safe cut edge.
5. Analyze using both $|V|$ and $|E|$. Writing only “$O(n)$” hides the graph representation and can turn a correct complexity claim into an ambiguous one.

The queue from BFS, stack from DFS, heap from Dijkstra and Prim, and DSU from Kruskal are supporting data structures. The graph algorithm is defined by the state and invariant maintained around those structures, not by the container alone.

**Practice.** [LeetCode 399 - Evaluate Division](https://leetcode.com/problems/evaluate-division/) is a useful synthesis exercise: model equations as weighted directed edges, then answer each query by graph traversal and accumulated path weight.
